# Homework 7. Time Series and JSONB for ML
## Task 1. Setup and Data Loading

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

import os
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text

# Start PostgreSQL 16
pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

# Check PostgreSQL version
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.5 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
# Download the official NYC Yellow Taxi Parquet file
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(DATA_URL, parquet_path)

print("File downloaded successfully.")
print(f"File size: {os.path.getsize(parquet_path) / 1024**2:.1f} MB")

File downloaded successfully.
File size: 57.3 MB


In [3]:
# Read only the required columns
columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count"
]

df = pd.read_parquet(parquet_path, columns=columns)

print(f"Original rows: {len(df):,}")

# Keep pickup timestamps from March 2024
df = df[
    (df["tpep_pickup_datetime"] >= "2024-03-01")
    & (df["tpep_pickup_datetime"] < "2024-04-01")
]

# Remove invalid fares, distances, and missing pickup zones
df = df[
    (df["fare_amount"] > 0)
    & (df["fare_amount"] < 500)
    & (df["trip_distance"] > 0)
    & (df["trip_distance"] < 100)
    & df["PULocationID"].notna()
]

print(f"Rows after cleaning: {len(df):,}")

# Take a reproducible sample across the full month
sample_size = min(500_000, len(df))
df = df.sample(n=sample_size, random_state=42).copy()

print(f"Final sample: {len(df):,} rows")
print("First pickup:", df["tpep_pickup_datetime"].min())
print("Last pickup:", df["tpep_pickup_datetime"].max())

Original rows: 3,582,628
Rows after cleaning: 3,439,911
Final sample: 500,000 rows
First pickup: 2024-03-01 00:00:03
Last pickup: 2024-03-31 23:59:51


In [4]:
# Load sampled trips into a PostgreSQL raw table
df.to_sql(
    "nyc_taxi_raw",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000
)

# Verify imported row count
raw_count_df = pd.read_sql(text("""
    SELECT COUNT(*) AS n_rows
    FROM nyc_taxi_raw
"""), engine)

display(raw_count_df)

,n_rows
0,500000


In [5]:
# Create daily aggregated taxi data
daily_sql = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features CASCADE;
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    tpep_pickup_datetime::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    ROUND(AVG(fare_amount)::NUMERIC, 2) AS avg_fare,
    ROUND(SUM(trip_distance)::NUMERIC, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    tpep_pickup_datetime::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

ALTER TABLE t7_hw_taxi_daily
    ADD CONSTRAINT chk_taxi_daily_trips
    CHECK (trip_count >= 0);

ALTER TABLE t7_hw_taxi_daily
    ADD CONSTRAINT chk_taxi_daily_fare
    CHECK (avg_fare >= 0);
"""

with engine.begin() as conn:
    conn.execute(text(daily_sql))

print("Daily taxi table created successfully.")

Daily taxi table created successfully.


In [6]:
# Verify daily time-series coverage
daily_summary_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT pickup_zone_id) AS pickup_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily
"""), engine)

display(daily_summary_df)

assert daily_summary_df.iloc[0]["total_rows"] > 100
assert daily_summary_df.iloc[0]["total_trips"] == len(df)

print("Daily aggregation verification passed.")

,total_rows,pickup_zones,first_day,last_day,total_trips
0,5445,250,2024-03-01,2024-03-31,500000


Daily aggregation verification passed.


In [7]:
# Check how many days are available for each pickup zone
zone_coverage_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        COUNT(*) AS days_with_trips,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
    ORDER BY days_with_trips DESC, total_trips DESC
    LIMIT 10
"""), engine)

display(zone_coverage_df)

,pickup_zone_id,days_with_trips,total_trips
0,161,31,23220
1,237,31,22138
2,132,31,21829
3,236,31,20539
4,162,31,17493
5,230,31,17200
6,186,31,16232
7,142,31,16205
8,138,31,15364
9,239,31,14428


### Результат

Було завантажено дані NYC Yellow Taxi за березень 2024 року та сформовано вибірку до 500 000 поїздок.

Перед завантаженням у PostgreSQL було видалено записи з некоректною вартістю, відстанню та відсутнім ідентифікатором pickup-зони.

На основі цих даних було створено таблицю `t7_hw_taxi_daily`. У ній для кожної зони та дня розраховано кількість поїздок, середню вартість і загальну відстань.

Після цього було перевірено кількість записів, часовий діапазон та кількість днів із поїздками для різних зон.

Отримана таблиця буде використовуватися для розрахунку часових ознак за допомогою window functions.

## Task 2. Time-Series ML Features

In [8]:
# Create a complete calendar for every pickup zone
calendar_sql = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features CASCADE;
DROP TABLE IF EXISTS t7_hw_taxi_calendar CASCADE;

CREATE TABLE t7_hw_taxi_calendar AS
WITH zones AS (
    SELECT DISTINCT pickup_zone_id
    FROM t7_hw_taxi_daily
),
calendar AS (
    SELECT
        generate_series(
            DATE '2024-03-01',
            DATE '2024-03-31',
            INTERVAL '1 day'
        )::DATE AS ts_day
)
SELECT
    z.pickup_zone_id,
    c.ts_day,
    COALESCE(d.trip_count, 0)::INTEGER AS trip_count,
    d.avg_fare,
    d.total_distance
FROM zones AS z
CROSS JOIN calendar AS c
LEFT JOIN t7_hw_taxi_daily AS d
    ON d.pickup_zone_id = z.pickup_zone_id
   AND d.ts_day = c.ts_day;

ALTER TABLE t7_hw_taxi_calendar
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
"""

with engine.begin() as conn:
    conn.execute(text(calendar_sql))

print("Calendar spine created successfully.")

Calendar spine created successfully.


In [9]:
# Verify complete daily coverage
calendar_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT pickup_zone_id) AS pickup_zones,
        COUNT(*) FILTER (WHERE trip_count = 0) AS zero_trip_days,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM t7_hw_taxi_calendar
"""), engine)

display(calendar_df)

assert calendar_df.iloc[0]["total_rows"] == 7750
assert calendar_df.iloc[0]["zero_trip_days"] == 2305

print("Calendar verification passed.")

,total_rows,pickup_zones,zero_trip_days,first_day,last_day
0,7750,250,2305,2024-03-01,2024-03-31


Calendar verification passed.


### 2.2. Lag-ознаки

Для розрахунку попередніх значень використано функцію `LAG()`.

Було створено три ознаки: кількість поїздок за 1, 7 та 30 днів до поточної дати.

У всіх функціях використано `PARTITION BY pickup_zone_id`, щоб історія кожної зони обчислювалася окремо.

На початку місяця деякі lag-значення будуть `NULL`, оскільки для них ще немає достатньої історії.

In [10]:
# Calculate 1-day, 7-day, and 30-day lag features
lag_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER w AS trips_lag_1d,
        LAG(trip_count, 7) OVER w AS trips_lag_7d,
        LAG(trip_count, 30) OVER w AS trips_lag_30d
    FROM t7_hw_taxi_calendar
    WINDOW w AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
"""), engine)

display(lag_df)

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,0,NaN,NaN,None
1,1,2024-03-02,0,0.0,NaN,None
2,1,2024-03-03,0,0.0,NaN,None
3,1,2024-03-04,0,0.0,NaN,None
4,1,2024-03-05,0,0.0,NaN,None
5,1,2024-03-06,0,0.0,NaN,None
6,1,2024-03-07,0,0.0,NaN,None
7,1,2024-03-08,0,0.0,0.0,None
8,1,2024-03-09,0,0.0,0.0,None
9,1,2024-03-10,0,0.0,0.0,None


In [11]:
# Check lag values on March 31 against historical dates
lag_check_df = pd.read_sql(text("""
    WITH lagged AS (
        SELECT
            pickup_zone_id,
            ts_day,
            LAG(trip_count, 30) OVER (
                PARTITION BY pickup_zone_id
                ORDER BY ts_day
            ) AS trips_lag_30d
        FROM t7_hw_taxi_calendar
    )
    SELECT
        l.pickup_zone_id,
        l.ts_day,
        l.trips_lag_30d,
        d.trip_count AS march_1_trips
    FROM lagged AS l
    JOIN t7_hw_taxi_calendar AS d
        ON d.pickup_zone_id = l.pickup_zone_id
       AND d.ts_day = DATE '2024-03-01'
    WHERE l.ts_day = DATE '2024-03-31'
    ORDER BY l.pickup_zone_id
    LIMIT 10
"""), engine)

display(lag_check_df)

assert (
    lag_check_df["trips_lag_30d"]
    == lag_check_df["march_1_trips"]
).all()

print("30-day lag verification passed.")

,pickup_zone_id,ts_day,trips_lag_30d,march_1_trips
0,1,2024-03-31,0,0
1,2,2024-03-31,0,0
2,3,2024-03-31,1,1
3,4,2024-03-31,26,26
4,6,2024-03-31,0,0
5,7,2024-03-31,11,11
6,8,2024-03-31,0,0
7,9,2024-03-31,1,1
8,10,2024-03-31,7,7
9,11,2024-03-31,1,1


30-day lag verification passed.


### 2.3. Rolling average за 7 днів

За допомогою `AVG() OVER` було розраховано середню вартість поїздок за останні 7 календарних днів.

У вікно входить поточний день і шість попередніх. Для цього використано `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW`.

Якщо в певний день не було поїздок у вибірці, значення `avg_fare` залишається `NULL` і не враховується при обчисленні середнього.

In [12]:
# Calculate rolling 7-day average fare
rolling_7d_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        ROUND(
            AVG(avg_fare) OVER w7,
            2
        ) AS rolling_avg_fare_7d,
        COUNT(avg_fare) OVER w7 AS days_with_fares
    FROM t7_hw_taxi_calendar
    WINDOW w7 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN
            INTERVAL '6 days' PRECEDING
            AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
"""), engine)

display(rolling_7d_df)

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_with_fares
0,1,2024-03-01,NaN,NaN,0
1,1,2024-03-02,NaN,NaN,0
2,1,2024-03-03,NaN,NaN,0
3,1,2024-03-04,NaN,NaN,0
4,1,2024-03-05,NaN,NaN,0
5,1,2024-03-06,NaN,NaN,0
6,1,2024-03-07,NaN,NaN,0
7,1,2024-03-08,NaN,NaN,0
8,1,2024-03-09,NaN,NaN,0
9,1,2024-03-10,NaN,NaN,0


### 2.4. Rolling volatility за 30 днів

Для оцінки коливань вартості поїздок було використано функцію `STDDEV_SAMP()`.

Вона розраховує стандартне відхилення середньої вартості поїздок за останні 30 календарних днів.

Якщо значення велике, це означає, що середня вартість поїздок у різні дні сильніше відрізнялася. Якщо значення мале, ціни були більш стабільними.

In [13]:
# Calculate rolling 30-day fare volatility
rolling_30d_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        ts_day,
        avg_fare,
        ROUND(
            STDDEV_SAMP(avg_fare) OVER w30,
            2
        ) AS rolling_std_fare_30d,
        COUNT(avg_fare) OVER w30 AS days_with_fares
    FROM t7_hw_taxi_calendar
    WINDOW w30 AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day::TIMESTAMP
        RANGE BETWEEN
            INTERVAL '29 days' PRECEDING
            AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
"""), engine)

display(rolling_30d_df)

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_with_fares
0,1,2024-03-01,NaN,NaN,0
1,1,2024-03-02,NaN,NaN,0
2,1,2024-03-03,NaN,NaN,0
3,1,2024-03-04,NaN,NaN,0
4,1,2024-03-05,NaN,NaN,0
5,1,2024-03-06,NaN,NaN,0
6,1,2024-03-07,NaN,NaN,0
7,1,2024-03-08,NaN,NaN,0
8,1,2024-03-09,NaN,NaN,0
9,1,2024-03-10,NaN,NaN,0


### 2.5. Expanding features

Для розрахунку накопичених значень було використано expanding window.

Ознака `cumulative_trips` показує загальну кількість поїздок від початку періоду до поточної дати.

Ознака `expanding_avg_trips` показує середню кількість поїздок за день за весь доступний період.

Для цього використано `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.

In [14]:
# Calculate cumulative trips and expanding daily average
expanding_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        SUM(trip_count) OVER we AS cumulative_trips,
        ROUND(
            AVG(trip_count) OVER we,
            2
        ) AS expanding_avg_trips
    FROM t7_hw_taxi_calendar
    WINDOW we AS (
        PARTITION BY pickup_zone_id
        ORDER BY ts_day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
    ORDER BY pickup_zone_id, ts_day
    LIMIT 20
"""), engine)

display(expanding_df)

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,0,0,0.00
1,1,2024-03-02,0,0,0.00
2,1,2024-03-03,0,0,0.00
3,1,2024-03-04,0,0,0.00
4,1,2024-03-05,0,0,0.00
5,1,2024-03-06,0,0,0.00
6,1,2024-03-07,0,0,0.00
7,1,2024-03-08,0,0,0.00
8,1,2024-03-09,0,0,0.00
9,1,2024-03-10,0,0,0.00


In [15]:
# Create a combined SQL view with all time-series features
features_view_sql = """
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        -- Lag features
        LAG(trip_count, 1) OVER w_lag AS trips_lag_1d,
        LAG(trip_count, 7) OVER w_lag AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_lag AS trips_lag_30d,

        -- Rolling 7-day average
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,

        -- Rolling 30-day volatility
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        -- Expanding features
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips

    FROM t7_hw_taxi_calendar

    WINDOW
        w_lag AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING
                      AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING
                      AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING
                     AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
"""

with engine.begin() as conn:
    conn.execute(text(features_view_sql))

print("Time-series feature view created successfully.")

Time-series feature view created successfully.


In [16]:
# Verify and preview the combined feature view
features_summary_df = pd.read_sql(text("""
    SELECT
        COUNT(*) AS feature_rows,
        COUNT(DISTINCT pickup_zone_id) AS pickup_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day
    FROM v_t7_hw_taxi_features
"""), engine)

display(features_summary_df)

assert features_summary_df.iloc[0]["feature_rows"] == 250

features_df = pd.read_sql(text("""
    SELECT *
    FROM v_t7_hw_taxi_features
    ORDER BY pickup_zone_id, ts_day
    LIMIT 10
"""), engine)

display(features_df)

,feature_rows,pickup_zones,first_day,last_day
0,250,250,2024-03-31,2024-03-31


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-31,0,NaN,NaN,0,0,0,80.000000,76.663843,7,0.225806
1,2,2024-03-31,0,NaN,NaN,0,0,0,NaN,22.768838,2,0.064516
2,3,2024-03-31,0,NaN,NaN,1,0,1,34.500000,14.235644,20,0.645161
3,4,2024-03-31,57,19.55,188.71,61,58,26,17.685714,3.147960,986,31.806452
4,6,2024-03-31,0,NaN,NaN,0,1,0,NaN,24.006753,6,0.193548
5,7,2024-03-31,17,25.36,91.77,13,16,11,25.032857,7.185690,302,9.741935
6,8,2024-03-31,0,NaN,NaN,0,0,0,NaN,NaN,1,0.032258
7,9,2024-03-31,0,NaN,NaN,0,0,1,7.010000,16.854485,6,0.193548
8,10,2024-03-31,3,31.28,20.63,5,7,7,45.602857,10.173211,190,6.129032
9,11,2024-03-31,0,NaN,NaN,0,1,1,37.200000,18.814391,15,0.483871


In [17]:
# Create next-day prediction labels from historical demand
next_day_df = pd.read_sql(text("""
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count AS observed_trips,
        LEAD(trip_count, 1) OVER (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ) AS next_day_trips
    FROM t7_hw_taxi_calendar
    WHERE pickup_zone_id = (
        SELECT MIN(pickup_zone_id)
        FROM t7_hw_taxi_calendar
    )
    ORDER BY ts_day
    LIMIT 10
"""), engine)

display(next_day_df)

,pickup_zone_id,ts_day,observed_trips,next_day_trips
0,1,2024-03-01,0,0
1,1,2024-03-02,0,0
2,1,2024-03-03,0,0
3,1,2024-03-04,0,0
4,1,2024-03-05,0,0
5,1,2024-03-06,0,0
6,1,2024-03-07,0,0
7,1,2024-03-08,0,0
8,1,2024-03-09,0,0
9,1,2024-03-10,0,0


### Результат

Для аналізу часових рядів було створено календар із повним набором дат для кожної pickup-зони.

Після цього розраховано lag-ознаки за 1, 7 та 30 днів, середню вартість поїздок за 7 днів, стандартне відхилення за 30 днів і накопичену кількість поїздок.

У всіх window functions використано `PARTITION BY pickup_zone_id`, щоб дані різних зон не змішувалися між собою.

У результаті створено VIEW `v_t7_hw_taxi_features`, який містить усі розраховані ознаки.

Оскільки використовувалися дані лише за березень, повна історія для `trips_lag_30d` є тільки на 31 березня. Тому після фільтрації отримано 250 рядків.

Для реальної ML-моделі бажано мати дані за довший період. Також потрібно враховувати момент прогнозування, щоб модель не використовувала інформацію з майбутнього.

## Task 3. JSONB Event Log

In [18]:
# Create a table for synthetic user events
events_ddl = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm CASCADE;
DROP TABLE IF EXISTS t7_hw_events CASCADE;

CREATE TABLE t7_hw_events (
    event_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT NOT NULL
        CHECK (event_type IN (
            'page_view',
            'click',
            'add_to_cart',
            'checkout',
            'recommendation_shown'
        )),
    payload JSONB NOT NULL
);
"""

with engine.begin() as conn:
    conn.execute(text(events_ddl))

print("JSONB events table created successfully.")

JSONB events table created successfully.


In [19]:
# Generate 50,000 synthetic events with different JSONB structures
insert_events_sql = """
INSERT INTO t7_hw_events (
    user_id,
    event_ts,
    event_type,
    payload
)
SELECT
    (n % 1000) + 1 AS user_id,

    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,

    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END AS event_type,

    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer',
                (ARRAY['google', 'direct', 'email', 'fb'])[(n % 4) + 1],
            'device',
                (ARRAY['mobile', 'desktop', 'tablet'])[(n % 3) + 1]
        )

        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device',
                (ARRAY['mobile', 'desktop'])[(n % 2) + 1]
        )

        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo', 'new', 'clearance'])[(n % 3) + 1]
            )
        )

        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method',
                (ARRAY['card', 'paypal', 'crypto'])[(n % 3) + 1]
        )

        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload

FROM generate_series(1, 50000) AS n;
"""

with engine.begin() as conn:
    conn.execute(text(insert_events_sql))
    conn.execute(text("ANALYZE t7_hw_events"))

print("50,000 synthetic events inserted successfully.")

50,000 synthetic events inserted successfully.


In [20]:
# Count events by event type
events_count_df = pd.read_sql(text("""
    SELECT
        event_type,
        COUNT(*) AS n_events
    FROM t7_hw_events
    GROUP BY event_type
    ORDER BY event_type
"""), engine)

display(events_count_df)

assert events_count_df["n_events"].sum() == 50000
assert len(events_count_df) == 5
assert (events_count_df["n_events"] == 10000).all()

print("Event distribution verification passed.")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


Event distribution verification passed.


In [21]:
# Display one example of each JSONB event type
samples_df = pd.read_sql(text("""
    SELECT DISTINCT ON (event_type)
        event_type,
        payload
    FROM t7_hw_events
    ORDER BY event_type, event_id
"""), engine)

display(samples_df)

,event_type,payload
0,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
2,click,"{'device': 'desktop', 'page_url': '/category/2..."
3,page_view,"{'url': '/product/6', 'device': 'tablet', 'ref..."
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### 3.1. JSONB Containment

**Business-question:** Які замовлення були оплачені карткою?

Для пошуку використано оператор `@>`, який перевіряє, чи містить JSONB потрібну пару ключа та значення.

У цьому прикладі шукаються події `checkout`, де `payment_method` дорівнює `card`.

In [22]:
# Find checkout events paid by card
containment_df = pd.read_sql(text("""
    SELECT
        event_id,
        user_id,
        event_ts,
        payload
    FROM t7_hw_events
    WHERE event_type = 'checkout'
      AND payload @> jsonb_build_object('payment_method', 'card')
    ORDER BY event_id
    LIMIT 10
"""), engine)

display(containment_df)

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


In [23]:
# Count checkout events by payment method
payment_df = pd.read_sql(text("""
    SELECT
        payload ->> 'payment_method' AS payment_method,
        COUNT(*) AS n_events
    FROM t7_hw_events
    WHERE event_type = 'checkout'
    GROUP BY payload ->> 'payment_method'
    ORDER BY n_events DESC, payment_method
"""), engine)

display(payment_df)

,payment_method,n_events
0,card,3334
1,crypto,3333
2,paypal,3333


### 3.2. Перевірка наявності ключів

Для пошуку JSONB-полів було використано оператори `?` та `?|`.

Перший перевіряє наявність одного ключа, а другий - хоча б одного з декількох ключів.

Це дозволяє знаходити події за структурою їхнього `payload`, навіть якщо значення полів різні.

In [24]:
# Find events containing the tags key
tags_df = pd.read_sql(text("""
    SELECT
        event_id,
        event_type,
        payload
    FROM t7_hw_events
    WHERE payload ? 'tags'
    ORDER BY event_id
    LIMIT 10
"""), engine)

display(tags_df)

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [25]:
# Find events containing experiment or model_version
model_keys_df = pd.read_sql(text("""
    SELECT
        event_id,
        event_type,
        payload
    FROM t7_hw_events
    WHERE payload ?| ARRAY['experiment', 'model_version']
    ORDER BY event_id
    LIMIT 10
"""), engine)

display(model_keys_df)

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### 3.3. JSONPath

Для роботи зі складнішими JSONB-даними було використано JSONPath.

За допомогою `jsonb_path_query()` можна отримати окремі значення з JSON-масивів. У цьому прикладі витягуються ID товарів, які були показані користувачам у рекомендаціях.

Функція `jsonb_path_exists()` використовується для перевірки умов. Наприклад, можна знайти оформлені замовлення із сумою понад 300.

In [26]:
# Extract product IDs from recommendation arrays
recommendations_df = pd.read_sql(text("""
    SELECT
        event_id,
        jsonb_path_query(
            payload,
            '$.recommendations[*]'
        ) AS recommended_product_id
    FROM t7_hw_events
    WHERE event_type = 'recommendation_shown'
    ORDER BY event_id
    LIMIT 15
"""), engine)

display(recommendations_df)

,event_id,recommended_product_id
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [27]:
# Find checkout events with totals greater than 300
expensive_checkouts_df = pd.read_sql(text("""
    SELECT
        event_id,
        user_id,
        payload ->> 'order_id' AS order_id,
        (payload ->> 'total')::NUMERIC(10, 2) AS total
    FROM t7_hw_events
    WHERE event_type = 'checkout'
      AND jsonb_path_exists(
          payload,
          '$.total ? (@ > 300)'
      )
    ORDER BY total DESC, event_id
    LIMIT 10
"""), engine)

display(expensive_checkouts_df)

,event_id,user_id,order_id,total
0,498,499,o_100498,548.0
1,998,999,o_100998,548.0
2,1498,499,o_101498,548.0
3,1998,999,o_101998,548.0
4,2498,499,o_102498,548.0
5,2998,999,o_102998,548.0
6,3498,499,o_103498,548.0
7,3998,999,o_103998,548.0
8,4498,499,o_104498,548.0
9,4998,999,o_104998,548.0


In [28]:
# Verify JSONB query results
assert not containment_df.empty
assert not tags_df.empty
assert (tags_df["event_type"] == "add_to_cart").all()

assert not model_keys_df.empty
assert (model_keys_df["event_type"] == "recommendation_shown").all()

assert not recommendations_df.empty
assert not expensive_checkouts_df.empty
assert (expensive_checkouts_df["total"] > 300).all()

print("All JSONB query checks passed.")

All JSONB query checks passed.


### Результат

Було створено таблицю `t7_hw_events`, яка містить 50 000 синтетичних подій п'яти різних типів.

Для кожного типу події використано свою структуру JSONB. Наприклад, для `checkout` зберігаються сума замовлення та спосіб оплати, а для `recommendation_shown` — інформація про модель та список рекомендованих товарів.

Після створення таблиці було виконано кілька JSONB-запитів.

Оператор `@>` дозволив знайти оплати карткою, `?` та `?|` - перевірити наявність ключів, а JSONPath - отримати рекомендовані товари та знайти замовлення із сумою понад 300.

Таким чином, JSONB дозволяє працювати з подіями різної структури без створення окремої таблиці для кожного типу події.

## Task 4. GIN Index and Expression Index - Performance Comparison

In [29]:
# Remove indexes to establish a baseline
drop_indexes_sql = """
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
"""

with engine.begin() as conn:
    conn.execute(text(drop_indexes_sql))

# Measure containment query performance without GIN
gin_query = """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
"""

gin_before_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + gin_query),
    engine
)

for line in gin_before_df["QUERY PLAN"]:
    print(line)

Aggregate  (cost=1756.60..1756.61 rows=1 width=8) (actual time=57.111..57.114 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=2640 width=0) (actual time=0.014..56.682 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.285 ms
Execution Time: 57.149 ms


In [30]:
# Create a GIN index for JSONB containment queries
create_gin_sql = """
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events
    USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
"""

with engine.begin() as conn:
    conn.execute(text(create_gin_sql))

# Check GIN index size
gin_size_df = pd.read_sql(text("""
    SELECT
        pg_size_pretty(
            pg_relation_size('idx_t7_events_payload_gin_pathops')
        ) AS gin_index_size
"""), engine)

display(gin_size_df)

,gin_index_size
0,1144 kB


In [31]:
# Measure containment query performance with GIN
gin_after_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + gin_query),
    engine
)

for line in gin_after_df["QUERY PLAN"]:
    print(line)

Aggregate  (cost=1136.45..1136.46 rows=1 width=8) (actual time=6.360..6.361 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=44.53..1123.32 rows=5253 width=0) (actual time=0.897..6.126 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..43.21 rows=5253 width=0) (actual time=0.680..0.680 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.329 ms
Execution Time: 6.410 ms


In [32]:
# Test GIN with another JSONB containment condition
referrer_plan_df = pd.read_sql(text("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT COUNT(*)
    FROM t7_hw_events
    WHERE payload @> jsonb_build_object('referrer', 'google')
"""), engine)

for line in referrer_plan_df["QUERY PLAN"]:
    print(line)

Aggregate  (cost=1071.09..1071.10 rows=1 width=8) (actual time=5.916..5.919 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=26.27..1064.69 rows=2561 width=0) (actual time=0.618..5.714 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..25.63 rows=2561 width=0) (actual time=0.443..0.445 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.252 ms
Execution Time: 5.967 ms


In [33]:
# Measure device lookup performance before expression index
device_query = """
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
"""

device_before_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + device_query),
    engine
)

for line in device_before_df["QUERY PLAN"]:
    print(line)

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=17.039..17.040 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.020..16.316 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.113 ms
Execution Time: 17.073 ms


In [34]:
# Create a B-tree index on the device field inside JSONB
create_device_index_sql = """
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
"""

with engine.begin() as conn:
    conn.execute(text(create_device_index_sql))

print("Expression B-tree index created successfully.")

Expression B-tree index created successfully.


In [35]:
# Measure device lookup performance after expression index
device_after_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + device_query),
    engine
)

for line in device_after_df["QUERY PLAN"]:
    print(line)

Aggregate  (cost=1241.31..1241.32 rows=1 width=8) (actual time=19.090..19.091 rows=1 loops=1)
  Buffers: shared hit=2 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.44..1220.61 rows=8278 width=0) (actual time=0.723..14.308 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=2 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.38 rows=8278 width=0) (actual time=0.526..0.527 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=18 read=1 dirtied=3
Planning Time: 1.299 ms
Execution Time: 19.131 ms


In [36]:
import re

# Extract total execution time from PostgreSQL EXPLAIN output
def get_execution_time(plan_df):
    for line in plan_df["QUERY PLAN"]:
        match = re.search(
            r"Execution Time:\s*([\d.]+)\s*ms",
            line
        )
        if match:
            return float(match.group(1))
    return None

index_comparison_df = pd.DataFrame({
    "Test": [
        "Containment before GIN",
        "Containment after GIN",
        "Device before B-tree",
        "Device after B-tree"
    ],
    "Execution Time (ms)": [
        get_execution_time(gin_before_df),
        get_execution_time(gin_after_df),
        get_execution_time(device_before_df),
        get_execution_time(device_after_df)
    ]
})

display(index_comparison_df)

,Test,Execution Time (ms)
0,Containment before GIN,57.149
1,Containment after GIN,6.410
2,Device before B-tree,17.073
3,Device after B-tree,19.131


In [37]:
# Summarize the scan methods used by PostgreSQL
def get_scan_methods(plan_df):
    plan_text = "\n".join(plan_df["QUERY PLAN"].astype(str))
    methods = [
        "Bitmap Index Scan",
        "Bitmap Heap Scan",
        "Index Only Scan",
        "Index Scan",
        "Seq Scan"
    ]
    return ", ".join(
        method for method in methods if method in plan_text
    ) or "Other"

scan_comparison_df = pd.DataFrame({
    "Test": index_comparison_df["Test"],
    "Scan Method": [
        get_scan_methods(gin_before_df),
        get_scan_methods(gin_after_df),
        get_scan_methods(device_before_df),
        get_scan_methods(device_after_df)
    ]
})

display(scan_comparison_df)

,Test,Scan Method
0,Containment before GIN,Seq Scan
1,Containment after GIN,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
2,Device before B-tree,Seq Scan
3,Device after B-tree,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"


### Результати перевірки індексів

Було виконано порівняння SQL-запитів до та після створення індексів.

Спочатку перевірено пошук JSONB-документів за допомогою оператора `@>`. Для цього створено GIN-індекс із `jsonb_path_ops`.

Потім було перевірено пошук за полем `device`. Для нього створено окремий B-tree індекс на виразі `payload ->> 'device'`.

Для кожного запиту використано `EXPLAIN (ANALYZE, BUFFERS)`, щоб порівняти час виконання та спосіб читання даних.

GIN-індекс підходить для пошуку за вмістом JSONB, а expression B-tree index - для порівняння конкретного текстового значення.

При цьому індекси потребують додаткового місця та можуть уповільнювати вставку й оновлення записів. Тому їх варто створювати для запитів, які дійсно часто використовуються.

### Аналіз результатів індексації

Було порівняно час виконання JSONB-запитів до та після створення індексів.

Для GIN-індексу час виконання зменшився з 57.149 мс до 6.410 мс, тобто приблизно у 8.9 раза. PostgreSQL змінив `Seq Scan` на `Bitmap Index Scan` та `Bitmap Heap Scan`. Розмір створеного індексу становить 1144 kB.

Для пошуку за полем `device` результат був іншим. До створення B-tree індексу запит виконувався 17.073 мс, а після - 19.131 мс. Хоча PostgreSQL почав використовувати індекс, швидкість не покращилася.

Можлива причина - велика кількість записів із `device = 'mobile'`, через що використання індексу не дає переваги.

Отже, індекси не завжди прискорюють запити. Їхня користь залежить від типу пошуку, кількості записів і того, яка частина таблиці відповідає умові.

## Task 5. JSONB vs Normalized Tables

In [38]:
# Create a normalized table for checkout events
checkout_ddl = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL,
    order_id TEXT NOT NULL,
    total NUMERIC(10, 2) NOT NULL,
    items_count INTEGER NOT NULL CHECK (items_count > 0),
    payment_method TEXT NOT NULL
        CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id',
    (payload ->> 'total')::NUMERIC(10, 2),
    (payload ->> 'items_count')::INTEGER,
    payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm(payment_method);

ANALYZE t7_hw_checkouts_norm;
"""

with engine.begin() as conn:
    conn.execute(text(checkout_ddl))

print("Normalized checkout table created successfully.")

Normalized checkout table created successfully.


In [39]:
# Compare checkout records in both data models
checkout_counts_df = pd.read_sql(text("""
    SELECT
        (SELECT COUNT(*)
         FROM t7_hw_events
         WHERE event_type = 'checkout') AS jsonb_rows,

        (SELECT COUNT(*)
         FROM t7_hw_checkouts_norm) AS normalized_rows
"""), engine)

display(checkout_counts_df)

assert checkout_counts_df.iloc[0]["jsonb_rows"] == 10000
assert checkout_counts_df.iloc[0]["normalized_rows"] == 10000

print("Checkout data verification passed.")

,jsonb_rows,normalized_rows
0,10000,10000


Checkout data verification passed.


In [40]:
# Measure aggregation performance using JSONB
jsonb_analysis_sql = """
SELECT
    payload ->> 'payment_method' AS payment_method,
    ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total,
    COUNT(*) AS n_orders
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
"""

jsonb_result_df = pd.read_sql(text(jsonb_analysis_sql), engine)
display(jsonb_result_df)

jsonb_explain_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + jsonb_analysis_sql),
    engine
)

for line in jsonb_explain_df["QUERY PLAN"]:
    print(line)

,payment_method,avg_total,n_orders
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


Limit  (cost=1965.87..1965.88 rows=3 width=72) (actual time=15.092..15.094 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1965.87..1979.62 rows=5499 width=72) (actual time=15.091..15.092 rows=3 loops=1)
        Sort Key: (round(avg(((payload ->> 'total'::text))::numeric), 2)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1798.57..1894.80 rows=5499 width=72) (actual time=15.050..15.081 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.80 rows=9918 width=132) (actual time=0.014..9.505 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.109 ms
Execution Time: 15.159 ms


In [41]:
# Measure aggregation performance using normalized columns
normalized_analysis_sql = """
SELECT
    payment_method,
    ROUND(AVG(total), 2) AS avg_total,
    COUNT(*) AS n_orders
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
"""

normalized_result_df = pd.read_sql(
    text(normalized_analysis_sql),
    engine
)

display(normalized_result_df)

normalized_explain_df = pd.read_sql(
    text("EXPLAIN (ANALYZE, BUFFERS) " + normalized_analysis_sql),
    engine
)

for line in normalized_explain_df["QUERY PLAN"]:
    print(line)

,payment_method,avg_total,n_orders
0,paypal,300.52,3333
1,card,300.50,3334
2,crypto,300.48,3333


Limit  (cost=269.07..269.08 rows=3 width=46) (actual time=5.091..5.094 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.07..269.08 rows=3 width=46) (actual time=5.089..5.091 rows=3 loops=1)
        Sort Key: (round(avg(total), 2)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=5.076..5.080 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.016..1.105 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.120 ms
Execution Time: 5.501 ms


In [42]:
# Extract execution times and compare both approaches
comparison_df = pd.DataFrame({
    "Method": ["JSONB", "Normalized"],
    "Execution Time (ms)": [
        get_execution_time(jsonb_explain_df),
        get_execution_time(normalized_explain_df)
    ],
    "Scan Method": [
        get_scan_methods(jsonb_explain_df),
        get_scan_methods(normalized_explain_df)
    ]
})

display(comparison_df)

# Verify that both queries return the same aggregated values
pd.testing.assert_frame_equal(
    jsonb_result_df.sort_values("payment_method").reset_index(drop=True),
    normalized_result_df.sort_values("payment_method").reset_index(drop=True),
    check_dtype=False
)

print("Both models return the same results.")

,Method,Execution Time (ms),Scan Method
0,JSONB,15.159,Seq Scan
1,Normalized,5.501,Seq Scan


Both models return the same results.


### Результат

Для подій `checkout` було створено окрему нормалізовану таблицю з типізованими колонками.

До неї перенесено 10 000 записів із JSONB event-log.

Після цього було виконано однаковий SQL-запит для обох варіантів зберігання. Запит розраховує кількість замовлень і середню суму покупки для кожного способу оплати.

JSONB дозволяє зберігати події різної структури в одній таблиці, але для числових обчислень потрібно додатково витягувати та перетворювати значення.

У нормалізованій таблиці поля вже мають відповідні типи даних, тому їх простіше перевіряти й використовувати в SQL-запитах.

### Аналіз порівняння JSONB та нормалізованої таблиці

Було порівняно один і той самий аналітичний запит для двох моделей зберігання даних: JSONB та нормалізованої таблиці.

У JSONB-версії запит виконувався 15.159 мс, а в нормалізованій таблиці - 5.501 мс. Отже, в цьому запуску нормалізований варіант був приблизно у 2.8 раза швидший.

В обох випадках PostgreSQL використав `Seq Scan`, тобто послідовне читання таблиці. Це означає, що різниця в швидкості пов’язана не зі способом доступу до рядків, а з обробкою самих значень.

У випадку JSONB потрібно було витягувати значення з `payload` і додатково перетворювати їх до числового типу. У нормалізованій таблиці поля вже зберігаються у відповідних типах даних, тому обчислення виконуються простіше.

Отже, JSONB зручний для гнучкого зберігання подій різної структури, але для частих аналітичних обчислень нормалізовані таблиці зазвичай ефективніші.

## 6. Reflection

У цьому домашньому завданні було поєднано дві різні задачі: побудову часових ознак і роботу з напівструктурованими даними у JSONB.

Для time-series частини найважливішим було правильно підготувати календарний ряд. Без calendar spine функції `LAG()` працювали б не з попередніми календарними днями, а просто з попередніми наявними записами. Після заповнення пропусків вдалося коректно розрахувати `lag`, rolling та expanding features для кожної pickup-зони окремо через `PARTITION BY pickup_zone_id`.

Для JSONB-частини було видно, що PostgreSQL добре підходить для гнучкого зберігання подій різної структури. Через одну таблицю можна зберігати різні типи подій без створення окремої схеми для кожного з них. Це зручно, якщо структура payload може змінюватися з часом.

Порівняння індексів показало, що їх ефект залежить від типу запиту. GIN-індекс для containment-запиту дав помітне покращення: час виконання зменшився з 57.149 мс до 6.410 мс. Але expression B-tree index для `payload ->> 'device'` не покращив результат у цьому запуску: запит навіть став трохи повільнішим. Отже, індекси потрібно створювати не «про всяк випадок», а під конкретний сценарій.

Порівняння JSONB та нормалізованої таблиці показало, що нормалізований варіант був швидший: 5.501 мс проти 15.159 мс. Причина в тому, що для JSONB треба додатково витягувати і перетворювати значення.

На практиці PostgreSQL JSONB достатньо, якщо потрібна помірна гнучкість і зручно поєднувати документи з реляційними даними. Але якщо дані повністю документоорієнтовані й схема часто змінюється, можна розглядати MongoDB. Якщо ж основний акцент саме на великих часових рядах, окремо варто дивитися на TimescaleDB.